In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()



In [ ]:
system_prompt = """
You are a consumer-rights analyst who reads Terms & Conditions and Privacy Policies
on behalf of ordinary people who do not have time to read them.

TASK
Analyse the document provided and report what matters to the user.

OUTPUT FORMAT (use exactly these markdown headings):
## 1. What data is collected
## 2. Who it is shared with
## 3. Money and cancellation (fees, auto-renewal, refunds, how to cancel)
## 4. Your rights (deleting data, closing your account, opting out)
## 5. Red flags
List each red flag as: **Short title**: plain explanation. Quote the exact clause in "quotation marks" (max 25 words).
## 6. Verdict
One of: SAFE / CAUTION / RISKY, followed by a one-sentence reason.

RULES
- Use plain language a teenager could understand. No legal jargon.
- Only report what is in the document. If something is not stated, write "Not stated".
- Never invent clauses or quotes. Every quote must appear in the document.
- Be specific. "May share data" is weak; "Shares data with advertising partners for targeting" is good.
- Do not give legal advice.
"""

In [ ]:
user_prompt = """
Please analyze the following terms:

"We may share your personal information with our partners and affiliates for marketing purposes. You may not cancel your subscription within the first 12 months. We reserve the right to change these terms at any time without notice."
"""

In [ ]:

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
response = client.chat.completions.create(
 model="gpt-4o-mini",
   messages=messages,
)









In [ ]:
print(response.choices[0].message.content)

In [ ]:
import requests

def fetch_html(url):
    headers = {"User-Agent": "Mozilla/5.0 (compatible; TCRedFlagBot/1.0)"}
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    return resp.text

In [ ]:
html = fetch_html("https://policies.google.com/terms")
print(len(html))
print(html[:500])

In [ ]:
from bs4 import BeautifulSoup

def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript", "aside"]):
        tag.decompose()
    root = soup.find("main") or soup.find("article") or soup.body or soup
    return root.get_text(separator="\n", strip=True)

In [ ]:
text = html_to_text(fetch_html("https://www.whatsapp.com/legal/privacy-policy"))
print(len(text))
print(text[:1000])

In [ ]:
MAX_CHARS = 30000

def trim(text, limit=MAX_CHARS):
    if len(text) > limit:
        print(f"Warning: document trimmed from {len(text)} to {limit} characters")
        return text[:limit]
    return text

In [ ]:
def analyse(text):
    text = trim(text)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Analyse this document:\n\n{text}"},
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2,
    )
    return response.choices[0].message.content

In [ ]:
def analyse(text):
    text = trim(text)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Analyse this document:\n\n{text}"},
    ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2,
    )
    return response.choices[0].message.content

In [ ]:
text = html_to_text(fetch_html("https://policies.google.com/terms"))
report = analyse(text)
print(report)

In [ ]:
from IPython.display import Markdown, display

report = analyse(text)
display(Markdown(report))

In [ ]:
url = "https://www.whatsapp.com/legal/privacy-policy"
text = html_to_text(fetch_html(url))

print(len(text))
if len(text) < 500:
    raise ValueError("Very little text found. The page may need JavaScript to load.")

report = analyse(text)
display(Markdown(report))

In [ ]:
import re

def check_quotes(report, source_text):
    quotes = re.findall(r'"([^"]{15,200})"', report)
    source_norm = " ".join(source_text.lower().split())
    return [(q, " ".join(q.lower().split()) in source_norm) for q in quotes]

for quote, ok in check_quotes(report, text):
    print(("✔ verified: " if ok else "✘ NOT FOUND: ") + quote[:90])